In [0]:
%sql
CREATE TABLE IF NOT EXISTS sales.gold.dim_customer (
    CustomerID STRING,
    FirstName STRING,
    LastName STRING,
    City STRING,
    Email STRING,

    HASHVALUE BIGINT,

    START_DATE DATE,
    END_DATE DATE,
    IS_CURRENT INT,

    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

#  Source
df = spark.read.format("delta").table("sales.silver.customer")

#  Add HASH
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("CustomerID"),
        col("FirstName"),
        col("LastName"),
        col("City"),
        col("Email")
    ))
)

In [0]:
# Add SCD2 columns in source
df_hash = df_hash.withColumn("START_DATE", current_date()) \
                 .withColumn("END_DATE", lit(None).cast("date")) \
                 .withColumn("IS_CURRENT", lit(1))


In [0]:
# Target
target = DeltaTable.forName(spark, "sales.gold.dim_customer")



In [0]:
# STEP A: EXPIRE OLD RECORDS
(
    target.alias("tgt")
    .merge(
        df_hash.alias("src"),
        "tgt.CustomerID = src.CustomerID AND tgt.IS_CURRENT = 1"
    )
    .whenMatchedUpdate(
        condition = "tgt.HASHVALUE != src.src_hash",
        set = {
            "END_DATE": current_date(),
            "IS_CURRENT": lit(0),
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks-updated")
        }
    )
    .execute()
)

In [0]:
# STEP B: INSERT NEW RECORDS
(
    target.alias("tgt")
    .merge(
        df_hash.alias("src"),
        "tgt.CustomerID = src.CustomerID AND tgt.IS_CURRENT = 1"
    )
    .whenNotMatchedInsert(
        values = {
            "CustomerID": "src.CustomerID",
            "FirstName": "src.FirstName",
            "LastName": "src.LastName",
            "City": "src.City",
            "Email": "src.Email",
            "HASHVALUE": "src.src_hash",
            "START_DATE": "src.START_DATE",
            "END_DATE": "src.END_DATE",
            "IS_CURRENT": "src.IS_CURRENT",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks")
        }
    )
    .execute()
)